# YOLOv8 Flag Classification Model Training Notebook

This notebook trains a **YOLOv8 Small Image Classifier** (`yolov8s-cls`) to classify flags into 254 country/institution categories, plus a background/negative class.

## Step 1: Install Dependencies
We install the `ultralytics` package.

In [ ]:
!pip install -U ultralytics

## Step 2: Locate Dataset
We scan `/kaggle/input` to locate the classification dataset.

In [ ]:
import os
import shutil
import random
from pathlib import Path

# Paths
src_dataset = Path("/kaggle/input/new-dataset-updated-296-cropped-mergedd/new-DATASET_updated_296_cropped_Merged")
src_bg_dataset = Path("/kaggle/input/drone-flag-classification-dataset-2026")
dst_dataset = Path("/kaggle/working/dataset")

# Reset destination
if dst_dataset.exists():
    shutil.rmtree(dst_dataset)
dst_dataset.mkdir(parents=True, exist_ok=True)

# Splits
(dst_dataset / "train").mkdir(exist_ok=True)
(dst_dataset / "val").mkdir(exist_ok=True)

# Copy/symlink flag classes
print("Splitting flag classes...")
classes = [c for c in src_dataset.iterdir() if c.is_dir()]
for c in classes:
    class_name = c.name
    # Create train and val class folders
    (dst_dataset / "train" / class_name).mkdir(exist_ok=True)
    (dst_dataset / "val" / class_name).mkdir(exist_ok=True)
    
    # Get all images
    imgs = list(c.glob("*.jpg")) + list(c.glob("*.png")) + list(c.glob("*.jpeg")) + list(c.glob("*.JPG")) + list(c.glob("*.PNG")) + list(c.glob("*.JPEG"))
    random.seed(42)
    random.shuffle(imgs)
    
    # 85/15 split
    split_idx = int(len(imgs) * 0.85)
    train_imgs = imgs[:split_idx]
    val_imgs = imgs[split_idx:]
    
    # Symlink train images
    for idx, img in enumerate(train_imgs):
        os.symlink(img, dst_dataset / "train" / class_name / f"{idx:05d}_{img.name}")
        
    # Symlink val images
    for idx, img in enumerate(val_imgs):
        os.symlink(img, dst_dataset / "val" / class_name / f"{idx:05d}_{img.name}")

# Copy/symlink background class
print("Copying background class...")
(dst_dataset / "train" / "background").mkdir(exist_ok=True)
(dst_dataset / "val" / "background").mkdir(exist_ok=True)

bg_train_src = src_bg_dataset / "train" / "background"
bg_val_src = src_bg_dataset / "val" / "background"

# Symlink train background images
if bg_train_src.exists():
    for img in bg_train_src.glob("*.jpg"):
        os.symlink(img, dst_dataset / "train" / "background" / img.name)
        
# Symlink val background images
if bg_val_src.exists():
    for img in bg_val_src.glob("*.jpg"):
        os.symlink(img, dst_dataset / "val" / "background" / img.name)

print("Dataset preparation complete!")
dataset_dir = "/kaggle/working/dataset"

## Step 3: Initialize Model
We load the pre-trained classification model weights (`yolov8s-cls.pt`).

In [ ]:
import torch
from ultralytics import YOLO

cuda_available = torch.cuda.is_available()
print(f"GPU (CUDA) Available: {cuda_available}")
if cuda_available:
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    
model = YOLO("yolov8s-cls.pt")
print("Loaded yolov8s-cls.pt base model.")

## Step 4: Train classification model
We train the model. We default to **15 epochs** with a batch size of **32** and image size of **128**.

In [ ]:
if dataset_dir:
    device_val = 0 if torch.cuda.is_available() else 'cpu'
    model.train(
        data=dataset_dir,
        epochs=15,
        imgsz=128,
        batch=32,
        device=device_val,
        workers=2,
        name="yolov8s_cls_flag",
        exist_ok=True
    )
else:
    print("Skipping training because dataset was not found.")

## Step 5: Compress Outputs for Download
We package the final weights (`best.pt`, `last.pt`) and logs into a single `.zip` file for easy download from Kaggle.

In [ ]:
import zipfile

results_dir = "/kaggle/working/runs/classify/yolov8s_cls_flag"
zip_path = "/kaggle/working/yolov8s_cls_flag_results.zip"

if os.path.exists(results_dir):
    print(f"Compressing results from {results_dir}...")
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(results_dir):
            for file in files:
                file_path = os.path.join(root, file)
                zipf.write(file_path, os.path.relpath(file_path, results_dir))
    print(f"Saved package to: {zip_path}")
else:
    print("Error: Training runs directory not found.")